# Практика 03. Линейные модели

**Версия:** 2026-09-24 (7b54111)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 02. План работы:

1. **Часть 1** — линейные модели своими руками на NumPy: нормальное уравнение, градиент и градиентный
   спуск, ridge-регрессия, логистическая регрессия. Каждая функция сверяется со scikit-learn.
2. **Часть 2** — линейная и логистическая регрессия из scikit-learn на данных о качестве вина: полиномиальные
   признаки и регуляризация, отбор признаков Lasso, классификация «хороших» вин.
3. **Часть 3** — эксперимент и выводы: масштабирование и градиентный спуск, выбор силы регуляризации,
   выбор порога классификации. Код здесь простой, оценивается объяснение.

В части 1 решения должны быть **без циклов** по объектам и признакам — операциями над матрицами.
Цикл по итерациям градиентного спуска, конечно, нужен.

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над матрицами."
    )


def make_regression_data(n=200, d=4, noise=0.5, seed=1):
    """Синтетическая линейная регрессия со стандартизированными признаками."""
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    w = r.normal(size=d)
    y = X @ w + 1.5 + noise * r.normal(size=n)
    return X, y


def make_classification_data(n=300, d=3, seed=2):
    """Синтетическая бинарная классификация, классы перекрываются (минимум log-loss конечен)."""
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    logits = X @ np.array([1.5, -1.0, 0.5])[:d] - 0.3
    y = (r.uniform(size=n) < 1 / (1 + np.exp(-logits))).astype(int)
    return X, y


print("Готово")

# Часть 1. Линейные модели своими руками

Обозначения как в лекции: $\mathbf{X} \in \mathbb{R}^{n \times d}$ — матрица объекты–признаки, $\mathbf{y}$ — вектор ответов,
модель $f(\mathbf{x}) = \mathbf{w}^{\top}\mathbf{x} + b$. Свободный член $b$ мы храним отдельно от весов $\mathbf{w}$.

## Задание 1.1. Нормальное уравнение

Добавим к $\mathbf{X}$ столбец единиц: $\tilde{\mathbf{X}} = [\mathbf{X}, \mathbf{1}]$, $\tilde{\mathbf{w}} = (\mathbf{w}, b)$. Минимум
среднеквадратичной ошибки удовлетворяет нормальному уравнению
$$
\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}\,\tilde{\mathbf{w}} = \tilde{\mathbf{X}}^{\top}\mathbf{y}.
$$
Напишите `fit_normal_equation(X, y)`, возвращающую `(w, b)`. Решайте систему функцией `np.linalg.solve`,
а не обращением матрицы (`np.linalg.inv`): так быстрее и точнее. Столбец единиц удобно добавить через
`np.column_stack`. Без циклов.

In [ ]:
def fit_normal_equation(X, y):
    """Веса w (вектор длины d) и свободный член b линейной регрессии по МНК."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.linear_model import LinearRegression

X_reg, y_reg = make_regression_data()
w, b = fit_normal_equation(X_reg, y_reg)
sk = LinearRegression().fit(X_reg, y_reg)
assert np.shape(w) == (4,), f"w должен быть вектором длины 4 (по числу признаков), получено {np.shape(w)}"
assert np.allclose(w, sk.coef_) and np.isclose(b, sk.intercept_), "Веса или свободный член не совпадают с LinearRegression"
residuals = y_reg - (X_reg @ w + b)
assert abs(residuals.sum()) < 1e-8, "Сумма остатков при наличии свободного члена должна быть равна нулю"
assert "inv(" not in inspect.getsource(fit_normal_equation), "Не обращайте матрицу: решайте систему np.linalg.solve"
assert_no_loops(fit_normal_equation)
print("OK")

## Задание 1.2. Градиент среднеквадратичной ошибки

Для $Q(\mathbf{w}, b) = \frac{1}{n}\sum_i (\mathbf{w}^{\top}\mathbf{x}_i + b - y_i)^2$
$$
\nabla_{\mathbf{w}} Q = \frac{2}{n}\mathbf{X}^{\top}(\mathbf{X}\mathbf{w} + b - \mathbf{y}), \qquad
\frac{\partial Q}{\partial b} = \frac{2}{n}\sum_{i=1}^n (\mathbf{w}^{\top}\mathbf{x}_i + b - y_i).
$$
Напишите `mse(X, y, w, b)` и `mse_gradient(X, y, w, b)`, возвращающую пару `(grad_w, grad_b)`. Без циклов.

In [ ]:
def mse(X, y, w, b):
    """Среднеквадратичная ошибка модели Xw + b."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def mse_gradient(X, y, w, b):
    """Градиент MSE: (вектор производных по w, производная по b)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
w0, b0 = rng.normal(size=4), 0.7
gw, gb = mse_gradient(X_reg, y_reg, w0, b0)
assert np.shape(gw) == (4,), f"grad_w должен иметь форму (4,), получено {np.shape(gw)}"
eps = 1e-6
num_gw = np.array([(mse(X_reg, y_reg, w0 + eps * e, b0) - mse(X_reg, y_reg, w0 - eps * e, b0)) / (2 * eps) for e in np.eye(4)])
num_gb = (mse(X_reg, y_reg, w0, b0 + eps) - mse(X_reg, y_reg, w0, b0 - eps)) / (2 * eps)
assert np.isclose(mse(X_reg, y_reg, w0, b0), np.mean((X_reg @ w0 + b0 - y_reg) ** 2)), "mse посчитана неверно"
assert np.allclose(gw, num_gw, atol=1e-5), "grad_w не совпадает с численной производной: проверьте множитель 2/n и транспонирование"
assert np.isclose(gb, num_gb, atol=1e-5), "grad_b не совпадает с численной производной"
assert_no_loops(mse_gradient)
print("OK")

## Задание 1.3. Градиентный спуск

Напишите `gradient_descent(X, y, lr, n_iter)`: начните с $\mathbf{w} = 0$, $b = 0$ и сделайте `n_iter` шагов
$$
\mathbf{w} \leftarrow \mathbf{w} - \eta\,\nabla_{\mathbf{w}} Q, \qquad b \leftarrow b - \eta\,\frac{\partial Q}{\partial b}.
$$
Верните `(w, b, history)`, где `history` — список из `n_iter` значений $Q$ **перед** каждым шагом. Если $Q$
стало бесконечным или не числом (спуск разошёлся), остановитесь досрочно: `np.isfinite` поможет.

In [ ]:
def gradient_descent(X, y, lr, n_iter):
    """Градиентный спуск для MSE. Возвращает (w, b, history)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
import warnings

w_gd, b_gd, hist = gradient_descent(X_reg, y_reg, lr=0.1, n_iter=500)
w_ne, b_ne = fit_normal_equation(X_reg, y_reg)
assert len(hist) == 500, f"history должна содержать 500 значений, получено {len(hist)}"
assert np.allclose(w_gd, w_ne, atol=1e-4) and np.isclose(b_gd, b_ne, atol=1e-4), "Градиентный спуск не сошёлся к решению нормального уравнения"
assert all(q_next <= q + 1e-12 for q, q_next in zip(hist, hist[1:])), "При небольшом шаге Q не должна возрастать"
assert np.isclose(hist[0], np.mean(y_reg ** 2)), "Первое значение history — Q в начальной точке w = 0, b = 0"
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _, _, hist_big = gradient_descent(X_reg, y_reg, lr=1.5, n_iter=500)
assert hist_big[-1] > hist_big[0] or not np.isfinite(hist_big[-1]), "При слишком большом шаге спуск должен расходиться"
print("OK")

## Задание 1.4. Ridge-регрессия

Как в scikit-learn, минимизируем $\left\lVert \mathbf{X}\mathbf{w} + b - \mathbf{y} \right\rVert^2 + \alpha\left\lVert \mathbf{w} \right\rVert^2$ (без деления на $n$).
Свободный член не регуляризуется. Удобный способ его учесть — центрировать данные: если
$\mathbf{X}_c = \mathbf{X} - \bar{\mathbf{x}}^{\top}$ (из каждого столбца вычтено среднее) и $\mathbf{y}_c = \mathbf{y} - \bar{y}$, то
$$
\mathbf{w}^* = \bigl(\mathbf{X}_c^{\top}\mathbf{X}_c + \alpha\mathbf{I}\bigr)^{-1}\mathbf{X}_c^{\top}\mathbf{y}_c, \qquad b^* = \bar{y} - \bar{\mathbf{x}}^{\top}\mathbf{w}^*.
$$
Напишите `fit_ridge(X, y, alpha)`, возвращающую `(w, b)`. Без циклов, систему решайте `np.linalg.solve`.

In [ ]:
def fit_ridge(X, y, alpha):
    """Ridge-регрессия в параметризации sklearn: веса w и свободный член b."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.linear_model import Ridge

X_shift = X_reg * [1, 10, 0.1, 3] + [5, -2, 0, 1]  # признаки в разных масштабах и не центрированы
for alpha in [0.0, 1.0, 50.0]:
    w, b = fit_ridge(X_shift, y_reg, alpha)
    sk = Ridge(alpha=alpha).fit(X_shift, y_reg)
    assert np.allclose(w, sk.coef_) and np.isclose(b, sk.intercept_), f"alpha={alpha}: результат не совпадает с Ridge из sklearn"
w_big, _ = fit_ridge(X_shift, y_reg, 1e6)
assert np.abs(w_big).max() < 0.05, "При огромном alpha веса должны сжаться почти до нуля"
assert_no_loops(fit_ridge)
print("OK")

## Задание 1.5. Логистическая регрессия

Модель $\hat{p}_i = \sigma(\mathbf{w}^{\top}\mathbf{x}_i + b)$, $\sigma(z) = 1/(1 + e^{-z})$, метки $y_i \in \{0, 1\}$, функция
потерь — log-loss
$$
Q(\mathbf{w}, b) = -\frac{1}{n}\sum_{i=1}^n \bigl[y_i\log\hat{p}_i + (1 - y_i)\log(1 - \hat{p}_i)\bigr],
$$
её градиент
$$
\nabla_{\mathbf{w}} Q = \frac{1}{n}\mathbf{X}^{\top}(\hat{\mathbf{p}} - \mathbf{y}), \qquad \frac{\partial Q}{\partial b} = \frac{1}{n}\sum_{i=1}^n (\hat{p}_i - y_i).
$$
Напишите `sigmoid(z)`, `log_loss(X, y, w, b)`, `log_loss_gradient(X, y, w, b)` и `fit_logreg(X, y, lr, n_iter)` —
градиентный спуск из нулевой точки, возвращающий `(w, b)`. Регуляризации нет. Без циклов, кроме цикла по итерациям.

In [ ]:
def sigmoid(z):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def log_loss(X, y, w, b):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def log_loss_gradient(X, y, w, b):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def fit_logreg(X, y, lr, n_iter):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss as sk_log_loss

assert np.isclose(sigmoid(0.0), 0.5) and np.allclose(sigmoid(np.array([-2.0, 2.0])).sum(), 1), "sigmoid: σ(0) = 0.5 и σ(-z) + σ(z) = 1"
X_clf, y_clf = make_classification_data()
w0, b0 = rng.normal(size=3), -0.2
assert np.isclose(log_loss(X_clf, y_clf, w0, b0), sk_log_loss(y_clf, sigmoid(X_clf @ w0 + b0))), "log_loss не совпадает с sklearn.metrics.log_loss"
gw, gb = log_loss_gradient(X_clf, y_clf, w0, b0)
eps = 1e-6
num_gw = np.array([(log_loss(X_clf, y_clf, w0 + eps * e, b0) - log_loss(X_clf, y_clf, w0 - eps * e, b0)) / (2 * eps) for e in np.eye(3)])
assert np.allclose(gw, num_gw, atol=1e-6), "Градиент по w не совпадает с численной производной"
w, b = fit_logreg(X_clf, y_clf, lr=1.0, n_iter=3000)
sk = LogisticRegression(penalty=None, tol=1e-10, max_iter=10000).fit(X_clf, y_clf)
assert np.allclose(w, sk.coef_[0], atol=1e-3) and np.isclose(b, sk.intercept_[0], atol=1e-3), (
    "Веса не совпадают с LogisticRegression(penalty=None): проверьте градиент и число итераций"
)
for f in (sigmoid, log_loss, log_loss_gradient):
    assert_no_loops(f)
print("OK")

# Часть 2. Линейные модели в scikit-learn: качество вина

Данные — 6497 португальских вин «винью верде», красных и белых (Cortez et al., 2009; UCI Wine Quality,
копия на OpenML, id 287). Признаки — 11 физико-химических измерений: кислотность, сахар, хлориды,
диоксид серы, плотность, pH, сульфаты, алкоголь. Целевая переменная `quality` — медианная оценка экспертов
от 0 до 10. Мы решим две задачи: **регрессию** оценки и **классификацию** «хорошее вино» (оценка не ниже 7,
около 20% вин).

In [ ]:
# @title Загрузка данных: Wine Quality (OpenML, id 287) и разбиение { display-mode: "form" }
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

wine = fetch_openml(data_id=287, as_frame=True, parser="auto").frame
FEATURES = [c for c in wine.columns if c != "quality"]
X_all = wine[FEATURES].to_numpy()
quality = wine["quality"].to_numpy()
good = (quality >= 7).astype(int)
X_train, X_test, q_train, q_test, g_train, g_test = train_test_split(
    X_all, quality, good, test_size=0.25, random_state=SEED, stratify=good
)
print(f"обучение: {X_train.shape}, тест: {X_test.shape}, доля хороших вин: {good.mean():.3f}")
wine[FEATURES].describe().T[["mean", "std", "min", "max"]]

## Задание 2.1. Линейная регрессия и полиномиальные признаки

Обучите три модели регрессии `quality` на `X_train` и заполните словарь `reg_results` значениями MSE на
тестовой выборке:

- `"linear"` — `StandardScaler` и `LinearRegression`;
- `"poly3"` — `PolynomialFeatures(3)`, `StandardScaler` и `LinearRegression`: все произведения признаков до
  третьей степени (364 признака);
- `"poly3_ridge"` — лучшая модель поиска по сетке `poly_ridge`: `PolynomialFeatures(3)`, `StandardScaler`,
  `Ridge()`; сетка `{"ridge__alpha": ALPHAS}`, `cv=KFOLD`, `scoring="neg_mean_squared_error"`,
  `return_train_score=True` (понадобится в части 3).

Сохраните также `poly_plain` — обученную модель `"poly3"`.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

ALPHAS = np.logspace(-3, 4, 15)
KFOLD = KFold(5, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучший alpha:", poly_ridge.best_params_)
pd.Series(reg_results, name="MSE на тесте").round(4)

In [ ]:
assert set(reg_results) == {"linear", "poly3", "poly3_ridge"}, f"Нужны ключи linear, poly3, poly3_ridge; получено {set(reg_results)}"
assert poly_ridge.scoring == "neg_mean_squared_error" and "mean_train_score" in poly_ridge.cv_results_, (
    "poly_ridge: scoring='neg_mean_squared_error' и return_train_score=True"
)
assert np.isclose(reg_results["poly3"], mean_squared_error(q_test, poly_plain.predict(X_test))), "MSE для poly3 посчитана неверно"
assert np.isclose(reg_results["poly3_ridge"], mean_squared_error(q_test, poly_ridge.predict(X_test))), "MSE для poly3_ridge посчитана неверно"
assert reg_results["poly3"] > reg_results["linear"], "Без регуляризации полиномиальная модель должна переобучиться и проиграть линейной на тесте"
assert reg_results["poly3_ridge"] < reg_results["poly3"], "Регуляризация должна исправить переобучение полиномиальной модели"
assert poly_ridge.best_params_["ridge__alpha"] >= 1, "Лучший alpha подозрительно мал — проверьте сетку"
assert "StandardScaler" in repr(poly_ridge.estimator), "poly_ridge: перед Ridge признаки нужно стандартизировать — штраф зависит от масштаба"
print("OK")

## Задание 2.2. Lasso: какие признаки лишние

Обучите на `X_train` модель `lasso`: `StandardScaler` и `Lasso(alpha=0.02)`. Сохраните её веса в
`pd.Series` `lasso_coef` с индексом `FEATURES`. Какие признаки Lasso обнулил? Обсудите это в части 3.

In [ ]:
from sklearn.linear_model import Lasso

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

lasso_coef.round(3).sort_values()

In [ ]:
assert isinstance(lasso_coef, pd.Series) and list(lasso_coef.index) == FEATURES, "lasso_coef: Series с индексом FEATURES"
assert np.allclose(lasso_coef.values, lasso[-1].coef_), "lasso_coef должен содержать веса обученной модели"
assert "StandardScaler" in repr(lasso), "lasso: перед Lasso признаки нужно стандартизировать — штраф зависит от масштаба"
assert (lasso_coef == 0).sum() >= 2, "При alpha=0.02 несколько весов должны обнулиться — проверьте масштабирование и alpha"
print("OK")

## Задание 2.3. Логистическая регрессия: хорошие вина

Подберите коэффициент регуляризации логистической регрессии для задачи «хорошее вино» (`g_train`):
`GridSearchCV` над конвейером `StandardScaler` + `LogisticRegression(max_iter=1000)`, сетка
`{"logisticregression__C": CS}`, `cv=SKFOLD`, `scoring="roc_auc"`. Результат — `logreg`, обученный на
`X_train`, `g_train`. Затем заполните `clf_results`: ROC AUC (по вероятностям) и F1 (по `predict`, то есть
при пороге 0.5) на тестовой выборке.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

CS = np.logspace(-4, 3, 15)
SKFOLD = StratifiedKFold(5, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучший C:", logreg.best_params_)
print({k: round(v, 3) for k, v in clf_results.items()})

In [ ]:
assert logreg.scoring == "roc_auc" and len(logreg.cv_results_["params"]) == len(CS), "logreg: scoring='roc_auc', сетка CS"
assert np.isclose(clf_results["roc_auc"], roc_auc_score(g_test, logreg.predict_proba(X_test)[:, 1])), (
    "roc_auc: нужны вероятности predict_proba(X_test)[:, 1] на тестовой выборке"
)
assert np.isclose(clf_results["f1_at_0.5"], f1_score(g_test, logreg.predict(X_test))), "f1_at_0.5: F1 по predict на тестовой выборке"
assert clf_results["roc_auc"] > 0.75, f"ROC AUC подозрительно низкий: {clf_results['roc_auc']:.3f}"
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. Масштабирование и градиентный спуск

Запустите свой `gradient_descent` (задание 1.3) для регрессии `quality` на **исходных** признаках `X_train`
с шагами из `LR_RAW` и на **стандартизированных** (`StandardScaler().fit_transform(X_train)`) с шагами из
`LR_SCALED`, по 500 итераций. Сохраните истории в словари `hist_raw` и `hist_scaled` (шаг → history) и
постройте все кривые на одном графике в логарифмическом масштабе по оси $Q$ (`plt.yscale("log")`).
Горизонтальной линией отметьте оптимум — MSE на обучении решения нормального уравнения `q_opt`.
Расходящийся спуск выдаёт предупреждения о переполнении; их можно подавить так же, как в проверке
задания 1.3 (`warnings.catch_warnings()`), а огромные значения перед построением графика — обрезать.

Посчитайте также **числа обусловленности** $\kappa = \lambda_{\max}/\lambda_{\min}$ матрицы $\frac{2}{n}\mathbf{X}^{\top}\mathbf{X}$
для исходных и стандартизированных признаков: `kappa_raw`, `kappa_scaled` (`np.linalg.eigvalsh` возвращает
собственные числа симметричной матрицы).

In [ ]:
LR_RAW = [1e-6, 1e-5, 1e-4]
LR_SCALED = [0.01, 0.1, 0.3]
X_train_scaled = StandardScaler().fit_transform(X_train)
q_opt = mse(X_train, q_train, *fit_normal_equation(X_train, q_train))

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert set(hist_raw) == set(LR_RAW) and set(hist_scaled) == set(LR_SCALED), "Нужны истории для всех шагов"
assert not np.isfinite(hist_raw[1e-4][-1]) or hist_raw[1e-4][-1] > hist_raw[1e-4][0], "На исходных признаках шаг 1e-4 должен расходиться"
assert hist_raw[1e-5][-1] > 2 * q_opt, "На исходных признаках за 500 итераций спуск не должен дойти до оптимума"
assert abs(hist_scaled[0.3][-1] - q_opt) < 1e-3, "На стандартизированных признаках шаг 0.3 должен дойти до оптимума"
assert kappa_raw > 1e6 and kappa_scaled < 1e3, f"Проверьте числа обусловленности: {kappa_raw:.3g}, {kappa_scaled:.3g}"
print("OK")

## Задание 3.2. Сила регуляризации

По `poly_ridge.cv_results_` постройте MSE на обучающих фолдах и на кросс-валидации в зависимости от `alpha`
(логарифмическая шкала по `alpha`). Обратите внимание: при `scoring="neg_mean_squared_error"` в результатах
лежит MSE **со знаком минус**. Сохраните массивы `train_mse` и `cv_mse` в порядке `ALPHAS`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(train_mse) == len(ALPHAS) and len(cv_mse) == len(ALPHAS), "train_mse и cv_mse — по значению на каждый alpha"
assert (np.asarray(train_mse) > 0).all() and (np.asarray(cv_mse) > 0).all(), "MSE положительна: не забудьте сменить знак"
assert train_mse[0] < train_mse[-1], "С ростом alpha ошибка на обучении должна расти"
assert 0 < np.argmin(cv_mse) < len(ALPHAS) - 1, "Минимум на кросс-валидации должен быть внутри сетки"
print("OK")

## Задание 3.3. Порог классификации

При пороге 0.5 F1 невысока. Подберите порог, максимизирующий F1, **без использования тестовой выборки**:
получите вероятности на обучающей выборке кросс-валидацией (`cross_val_predict(logreg.best_estimator_,
X_train, g_train, cv=SKFOLD, method="predict_proba")[:, 1]`), переберите пороги из `THRESHOLDS` и выберите
лучший по F1 — `best_threshold`. Затем посчитайте F1 на тесте при этом пороге — `f1_tuned`.

In [ ]:
from sklearn.model_selection import cross_val_predict

THRESHOLDS = np.linspace(0.05, 0.95, 91)
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print(f"лучший порог: {best_threshold:.2f}, F1 на тесте: {f1_tuned:.3f} (при 0.5: {clf_results['f1_at_0.5']:.3f})")

In [ ]:
assert best_threshold in THRESHOLDS, "best_threshold должен быть одним из THRESHOLDS"
assert best_threshold < 0.5, "При доле хороших вин около 20% лучший по F1 порог должен быть ниже 0.5"
assert np.isclose(f1_tuned, f1_score(g_test, (logreg.predict_proba(X_test)[:, 1] >= best_threshold).astype(int))), (
    "f1_tuned: F1 на тестовой выборке при найденном пороге"
)
assert f1_tuned > clf_results["f1_at_0.5"], "Подобранный порог должен улучшить F1 на тесте"
print("OK")

## Задание 3.4. Выводы

Ответьте на вопросы, опираясь на свои графики и числа. Ответ на каждый вопрос — 2–4 предложения.

1. Почему нормальное уравнение решают через `np.linalg.solve`, а не через обращение матрицы? Что случилось
   бы, если бы среди признаков был, например, признак «общая кислотность» = `fixed.acidity` + `volatile.acidity`?
2. Почему на исходных признаках градиентный спуск либо расходится, либо почти не сдвигается, а на
   стандартизированных быстро сходится? Используйте числа обусловленности из задания 3.1.
3. Объясните результаты задания 2.1 и график задания 3.2: почему полиномиальная модель без регуляризации
   хуже линейной на тесте, как ведут себя ошибки на обучении и на кросс-валидации с ростом `alpha`?
   Свяжите со смещением и разбросом.
4. Какие признаки обнулил Lasso (задание 2.2)? Почему именно они могли оказаться «лишними»? Означает ли
   нулевой вес, что признак не связан с качеством вина?
5. Почему F1 при пороге 0.5 невысока, хотя ROC AUC приличный? Почему порог нужно подбирать на обучающей
   выборке (кросс-валидацией), а не на тестовой?
6. Посмотрите на веса логистической регрессии `logreg.best_estimator_[-1].coef_` (признаки стандартизированы).
   Какие признаки сильнее всего повышают и понижают шансы, что вино хорошее? Как интерпретировать вес
   $w_j$ через шансы $\hat{p}/(1 - \hat{p})$?

*Ваш ответ:*